In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
from pyproj import Transformer
import cartopy as ct
import matplotlib.pyplot as plt

In [ ]:
# CERF data CRS to WGS84 CRS
cerf_crs = 'ESRI:102003'
to_crs = 'EPSG:4326'
transformer = Transformer.from_crs(cerf_crs, to_crs)
def crs_transform(row):
    lat, lon = transformer.transform(row['xcoord'], row['ycoord'])
    row['lat'] = lat
    row['lon'] = lon
    return row

In [ ]:
# translate CERF tech names into GPPD/Tethys tech categories
tech_mapping = {
    'hydro': 'Hydro',
    'gas (steam)': 'Gas',
    'coal (conv pulv) (pre_1970)': 'Coal',
    'coal (conv pulv) (1970s)': 'Coal',
    'gas (CC)': 'Gas',
    'gas (CT)': 'Gas',
    'refined liquids (CT)': 'Oil',
    'Gen_II_LWR': 'Nuclear',
    'coal (conv pulv) (1980s)': 'Coal',
    'coal (conv pulv) (2010s)': 'Coal',
    'solar_PV': 'Solar',
    'coal (conv pulv) (1990s)': 'Coal',
    'coal (conv pulv) (2000s)': 'Coal',
    'geothermal': 'Geothermal',
    'biomass (conv)': 'Biomass',
    'coal (conv pulv)': 'Coal',
    'refined liquids (steam)': 'Oil',
    'coal (IGCC) (2010s)': 'Coal',
    'refined liquids (CC)': 'Oil',
    'coal (IGCC) (1990s)': 'Coal',
    'solar_CSP': 'Solar',
    'wind_onshore': 'Wind',
}

In [ ]:
# IM3 initial power plants file; also used by CERF
# On PIC at: /rcfs/projects/im3/exp_b/exp_b_multi_model_coupling_west/models/cerf/data/power_plant_data/power_plant_locations_csv/power_plant_locations.csv
file_im3_plants = './power_plant_locations.csv'

In [ ]:
# Global Power Plants Database
# https://datasets.wri.org/dataset/globalpowerplantdatabase
file_gppd_plants = './data/powerplants/global_power_plant_database.csv'

In [ ]:
# previously used Tethys plants file
# only used to provide the grid and study the necessary shape
file_tethys_plants = './data/powerplants/plants.nc'

In [ ]:
# load IM3 initial plants for 2015
# map tech names
im3_plants = pd.read_csv(
    file_im3_plants
).apply(
    crs_transform,
    axis=1
)[[
    'region_name', 'unit_size_mw', 'tech_name', 'lat', 'lon'
]]
im3_plants['tech_name'] = im3_plants.tech_name.map(tech_mapping)
im3_geo = gpd.GeoDataFrame(
    im3_plants,
    geometry=gpd.points_from_xy(im3_plants.lon, im3_plants.lat), crs="EPSG:4326"
)

In [ ]:
# confirm all techs have been mapped
if len(im3_plants[im3_plants.tech_name.isna()]) > 0:
    raise Exception(f'''
    
    Missing technology mapping:
    
    {im3_plants[im3_plants.tech_name.isna()]}
    
    ''')

In [ ]:
# load Global Power Plant Database
gppd_plants = pd.read_csv(file_gppd_plants)

In [ ]:
# extract the Tethys grid
tethys_plants = xr.open_dataset(file_tethys_plants).load()
tethys_points = tethys_plants.isel(year=0, drop=True)[['lat', 'lon']].to_dataframe().reset_index()
tethys_points = gpd.GeoDataFrame(
    tethys_points,
    geometry=gpd.points_from_xy(tethys_points.lon, tethys_points.lat), crs="EPSG:4326"
)

In [ ]:
# create geodataframe from GPPD data
gppd_geo = gppd_plants[['country', 'capacity_mw', 'latitude', 'longitude', 'primary_fuel']]
gppd_geo = gpd.GeoDataFrame(
    gppd_geo,
    geometry=gpd.points_from_xy(gppd_geo.longitude, gppd_geo.latitude), crs="EPSG:4326"
)
# exclude the CONUS plants (keeping Alaska and Hawaii)
min_lat = im3_plants.lat.min()
max_lat = im3_plants.lat.max()
min_lon = im3_plants.lon.min()
max_lon = im3_plants.lon.max()
gppd_geo = gppd_geo[
    (gppd_geo.country != 'USA') |
    (((gppd_geo.latitude <= min_lat) |
    (gppd_geo.latitude >= max_lat)) &
    ((gppd_geo.longitude <= min_lon) |
    (gppd_geo.longitude >= max_lon)))
]

In [ ]:
# combine the IM3 CONUS plants with the rest of the world GPPD plants
combined_plants = pd.concat([
    im3_geo,
    gppd_geo.rename(columns={
        'country': 'region_name',
        'capacity_mw': 'unit_size_mw',
        'latitude': 'lat',
        'longitude': 'lon',
        'primary_fuel': 'tech_name',
    }),
], ignore_index=True)

In [ ]:
# spatial join plants to the tethys grid
# aggregate plant capacity by tech to tethys grid cells
# pivot table on tech_name and create an xarray dataset in the form expected by Tethys
aggregated_plants = combined_plants.sjoin_nearest(
    tethys_points.rename(columns={'lat': 'tethys_lat', 'lon': 'tethys_lon'}),
    how='left',
    max_distance=1,
).groupby([
    'tech_name', 'tethys_lat', 'tethys_lon'
]).unit_size_mw.sum().reset_index().rename(columns={
    'tethys_lat': 'lat',
    'tethys_lon': 'lon',
})
aggregated_plants = xr.Dataset.from_dataframe(
    aggregated_plants.merge(
        tethys_points,
        on=['lat', 'lon'],
        how='right'
    ).pivot(index=['lat', 'lon'], columns='tech_name', values='unit_size_mw').drop(columns=[np.nan])
).rio.set_crs(to_crs).expand_dims(year=[2015])

In [ ]:
aggregated_plants

In [ ]:
for v in list(aggregated_plants.data_vars):
    aggregated_plants[v].attrs['units'] = 'aggregated MW capacity'

In [ ]:
aggregated_plants.to_netcdf('gppd_im3_tethys_plants.nc')

In [ ]:
# plot the new merged dataset for a plant type
crs = ct.crs.PlateCarree()
fig = plt.figure(figsize=(10, 5), dpi=150, layout='tight')
ax = plt.axes(projection=crs, frameon=False)
ax.coastlines()
aggregated_plants.sel(year=2015, drop=True).Biomass.plot(ax=ax)
# aggregated_plants.sel(year=2015, drop=True).where(
#     (aggregated_plants.lat >= min_lat) &
#     (aggregated_plants.lat <= max_lat) &
#     (aggregated_plants.lon >= min_lon) &
#     (aggregated_plants.lon <= max_lon),
#     drop=True
# ).Biomass.plot(ax=ax)

In [ ]:
# plot the old dataset for comparison
crs = ct.crs.PlateCarree()
fig = plt.figure(figsize=(10, 5), dpi=150, layout='tight')
ax = plt.axes(projection=crs, frameon=False)
ax.coastlines()
tethys_plants.sel(year=2015, drop=True).Biomass.plot(ax=ax)
# tethys_plants.sel(year=2015, drop=True).where(
#     (tethys_plants.lat >= min_lat) &
#     (tethys_plants.lat <= max_lat) &
#     (tethys_plants.lon >= min_lon) &
#     (tethys_plants.lon <= max_lon),
#     drop=True
# ).Biomass.plot(ax=ax)